# Exp-1 — BGE-M3 dense retrieval, statute corpus only

**Goal.** Measure recall@{50, 200, 500, 1000} on the val set (10 queries) for statute golds, using BGE-M3 dense embeddings. Tests both the raw English query and the German-translated query.

**Kill criterion.** If `recall@500` on statutes < 0.4 with either query form, BGE-M3 alone is insufficient and Exp-2 (HyDE) becomes mandatory before touching the court corpus.

**Inputs expected on Drive (`/content/drive/MyDrive/swiss_law/`):**
- `laws_de.csv` (70 MB)
- `val.csv`
- `val_translated_de.pkl` (dict: query_id → German translation)

Upload these once from `e:/llm_swiss_law/llm-agentic-legal-information-retrieval/` and `e:/llm_swiss_law/fresh_pipeline/artifacts/`.

**Outputs written to Drive:**
- `artifacts/laws_bgem3.npy` — (175933, 1024) float16 embeddings, cached for reuse by Exp-2/3.
- `artifacts/exp_A1_report.json` — recall numbers + per-query breakdown.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- Cell 1. Install deps ---
!pip install -q FlagEmbedding pandas numpy transformers==4.44.2

import os
print("Restarting runtime...")
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [1]:
# --- Cell 2. Mount Drive & set paths ---
from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle, re, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
print('Files on Drive:', sorted(p.name for p in ROOT.iterdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files on Drive: ['artifacts', 'court_considerations.csv', 'laws_de.csv', 'sample_submission.csv', 'test.csv', 'test_translated_de.pkl', 'train.csv', 'val.csv', 'val_translated_de.pkl']


In [2]:
# --- Cell 3. Load data ---
val = pd.read_csv(ROOT / 'val.csv')
laws = pd.read_csv(ROOT / 'laws_de.csv')
with open(ROOT / 'val_translated_de.pkl', 'rb') as f:
    val_de = pickle.load(f)

assert len(val) == 10, len(val)
assert {r.query_id for r in val.itertuples()} <= set(val_de), 'missing translations'

# Build doc strings: include citation so exact-match code mentions help.
docs = (laws['citation'].fillna('') + ' | ' +
        laws['title'].fillna('')     + ' | ' +
        laws['text'].fillna('')).tolist()
cits = laws['citation'].tolist()
print(f'laws: {len(docs)} docs | avg chars={int(np.mean([len(d) for d in docs]))}')

laws: 175933 docs | avg chars=408


In [3]:
# --- Cell 4. Load BGE-M3 ---
from FlagEmbedding import BGEM3FlagModel
import torch
assert torch.cuda.is_available(), 'need GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

In [4]:
# --- Cell 5. Encode corpus (cached) ---
EMB_PATH = ART / 'laws_bgem3.npy'
if EMB_PATH.exists():
    doc_emb = np.load(EMB_PATH)
    print('loaded cached embeddings:', doc_emb.shape, doc_emb.dtype)
else:
    t0 = time.time()
    out = model.encode(docs,
                       batch_size=64,
                       max_length=512,
                       return_dense=True,
                       return_sparse=False,
                       return_colbert_vecs=False)
    doc_emb = out['dense_vecs'].astype(np.float16)
    np.save(EMB_PATH, doc_emb)
    print(f'encoded {len(docs)} docs in {time.time() - t0:.0f}s -> {doc_emb.shape}')

pre tokenize: 100%|██████████| 2749/2749 [00:06<00:00, 401.20it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 2749/2749 [01:00<00:00, 45.11it/s]


encoded 175933 docs in 70s -> (175933, 1024)


In [5]:
# --- Cell 6. Encode val queries (EN and DE) ---
en_queries = [r.query for r in val.itertuples()]
de_queries = [val_de[r.query_id] for r in val.itertuples()]

q_en = model.encode(en_queries, batch_size=8, max_length=2048,
                    return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
q_de = model.encode(de_queries, batch_size=8, max_length=2048,
                    return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
print('q_en', q_en.shape, 'q_de', q_de.shape)

Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00, 56.79it/s]

q_en (10, 1024) q_de (10, 1024)


In [6]:
# --- Cell 7. Score and evaluate ---
# Matmul val x corpus (10 x 175k) in one shot — fine on GPU.
doc_emb_f32 = doc_emb.astype(np.float32)

def topk_indices(q, k):
    sims = q @ doc_emb_f32.T          # (N_q, N_doc)
    # partial sort: get top-k indices per row
    idx = np.argpartition(-sims, k - 1, axis=1)[:, :k]
    # refine ordering within each top-k
    rows = np.arange(idx.shape[0])[:, None]
    order = np.argsort(-sims[rows, idx], axis=1)
    return idx[rows, order]

def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def is_statute(c):
    return not (c.startswith('BGE ') or re.match(r'\d[A-Z]_', c) or re.match(r'[A-Z]\d[A-Z]_', c))

def recall_table(q_vecs, label, ks=(50, 200, 500, 1000)):
    per_q = []
    idx_all = {k: topk_indices(q_vecs, k) for k in ks}
    for i, row in enumerate(val.itertuples()):
        gold_all = set(parse(row.gold_citations))
        gold_stat = {c for c in gold_all if is_statute(c)}
        entry = {'query_id': row.query_id, 'n_gold_stat': len(gold_stat)}
        for k in ks:
            retrieved = {cits[j] for j in idx_all[k][i]}
            entry[f'stat_hit@{k}'] = len(gold_stat & retrieved)
        per_q.append(entry)
    agg = {f'stat_recall@{k}': sum(p[f'stat_hit@{k}'] for p in per_q)
                              / max(1, sum(p['n_gold_stat'] for p in per_q)) for k in ks}
    print(f'=== {label} ===')
    for k, v in agg.items():
        print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

report = {'en_query': recall_table(q_en, 'EN query'),
          'de_query': recall_table(q_de, 'DE query (translated)')}

=== EN query ===
  stat_recall@50 = 0.074
  stat_recall@200 = 0.121
  stat_recall@500 = 0.215
  stat_recall@1000 = 0.302
=== DE query (translated) ===
  stat_recall@50 = 0.067
  stat_recall@200 = 0.114
  stat_recall@500 = 0.168
  stat_recall@1000 = 0.228


In [7]:
# --- Cell 8. Save report ---
report['meta'] = {
    'model': 'BAAI/bge-m3',
    'corpus': 'laws_de.csv',
    'n_docs': len(docs),
    'n_queries': len(val),
    'kill_criterion': 'recall@500 >= 0.40 on statutes (either query form)'
}
out_path = ART / 'exp_A1_report.json'
with open(out_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print('saved:', out_path)

best_de = report['de_query']['agg']['stat_recall@500']
best_en = report['en_query']['agg']['stat_recall@500']
print(f"verdict: best recall@500 statutes = {max(best_de, best_en):.3f} "
      f"(threshold 0.40 -> {'PASS' if max(best_de, best_en) >= 0.4 else 'FAIL -> need HyDE'})")

saved: /content/drive/MyDrive/swiss_law/data/artifacts/exp_A1_report.json
verdict: best recall@500 statutes = 0.215 (threshold 0.40 -> FAIL -> need HyDE)
